# Audit complet de Tears of Steel (film live‑action)

Ce notebook télécharge le film libre `Tears of Steel` (12 minutes) et lance un audit exhaustif avec le pipeline CineInfini.

**Durée estimée** : environ 6‑8 minutes sur GPU, 20‑30 minutes sur CPU.

In [ ]:
# Installation
!pip install cineinfini-audit -q

In [ ]:
from pathlib import Path
from cineinfini import adaptive_multi_stage_audit, set_global_paths, CONFIG
from cineinfini.io.download import download_video

# Configuration
BASE_DIR = Path("/content/drive/MyDrive/cineinfini_data")  # à adapter selon votre Drive
BASE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"
BENCHMARK_DIR = BASE_DIR / "benchmark"
set_global_paths(MODELS_DIR, REPORTS_DIR, BENCHMARK_DIR)

# Téléchargement de Tears of Steel (si absent)
video_path = BASE_DIR / "videos" / "tears_of_steel.mp4"
if not video_path.exists():
    download_video(
        "https://download.blender.org/demo/movies/ToS/tears_of_steel_1080p.mov",
        "tears_of_steel",
        video_path.parent,
        is_zip=False
    )
else:
    print(f"Vidéo déjà présente : {video_path}")

In [ ]:
# Forcer l’analyse de toute la vidéo (12 minutes)
CONFIG["max_duration_s"] = 1000  # > durée réelle (734s)
CONFIG["narrative_coherence"] = True
CONFIG["n_frames_per_shot"] = 16
CONFIG["parallel_shots"] = True
CONFIG["max_workers"] = 4

print("Lancement de l’audit complet de Tears of Steel...")
report_dir = adaptive_multi_stage_audit(str(video_path), force_full_video=True)
print(f"Audit terminé. Rapport dans : {report_dir}")

In [ ]:
# Afficher le tableau de bord
from IPython.display import display, Markdown
dashboard_path = Path(report_dir) / "dashboard.md"
if dashboard_path.exists():
    with open(dashboard_path, "r") as f:
        display(Markdown(f.read()))
else:
    print("Dashboard non généré. Vérifiez les logs précédents.")